$$
J(\theta) = \mathbb{E}[R] - \beta D_{KL}(\pi_\theta || \pi_{ref})
$$

- 熵正则化的策略优化（Entropy-Regularized）
    - $\begin{aligned}
D_{KL}(\pi_{\theta} \| \pi_{ref}) &= \sum \pi_{\theta}(x) \log \frac{\pi_{\theta}(x)}{\pi_{ref}(x)} \\
&= \sum \pi_{\theta}(x) \log \pi_{\theta}(x) - \sum \pi_{\theta}(x) \log \pi_{ref}(x) \\
&= \underbrace{-H(\pi_{\theta})}_{\text{负熵}} + \underbrace{H(\pi_{\theta}, \pi_{ref})}_{\text{交叉熵}}
\end{aligned}$
    - $J(\theta) = \underbrace{\mathbb{E}[R(x)]}_{\text{最大化奖励}} + \underbrace{\beta H(P_{\theta})}_{\text{最大化熵}} - \underbrace{\beta H(P_{\theta}, P_{ref})}_{\text{锚定原模型}}$
- $L(\theta) = -(R(x) - \beta \cdot KL(x))$，只要对 $L(\theta)$ 求导 backward()，不就是对 $J(\theta)$ 求导吗？
- Loss 是 Objective 的“梯度估计器”（Gradient Estimator），而不是 Objective 本身。在深度学习（监督学习）中，Loss 就是 Objective（比如最小化 MSE，Loss 就是 MSE）。但在强化学习（RL）中，Loss 是一个为了凑出正确梯度而构造的“替身函数”。

$$
\nabla_\theta \mathbb{E}_{x \sim \pi_\theta} [f(x)] \neq \mathbb{E}_{x \sim \pi_\theta} [\nabla_\theta f(x)]
$$
- 期望的算符 $\mathbb{E}$ 本身依赖于参数 $\theta$。对参数 $\theta$ 求导时，不仅函数 $f(x)$ 的值可能发生变化（如果 $f$ 显式依赖 $\theta$），采样分布 $\pi_\theta(x)$ 也会发生变化。等式右边漏掉了“分布变化带来的梯度”。

$$
J(\theta) = \mathbb{E}_{x \sim \pi_\theta} [f(x)] = \int \pi_\theta(x) f(x) \, dx
$$
- $\nabla_\theta J(\theta) = \nabla_\theta \int \pi_\theta(x) f(x) \, dx = \int \nabla_\theta (\pi_\theta(x) f(x)) \, dx$
    - $\nabla_\theta (\pi_\theta(x) f(x)) = \underbrace{(\nabla_\theta \pi_\theta(x)) f(x)}_{\text{分布变化的贡献}} + \underbrace{\pi_\theta(x) (\nabla_\theta f(x))}_{\text{函数变化的贡献}}$

$$
\begin{aligned}
\nabla_\theta \mathbb{E}_{x \sim \pi_\theta} [f(x)] &= \int (\nabla_\theta \pi_\theta(x)) f(x) \, dx + \int \pi_\theta(x) (\nabla_\theta f(x)) \, dx \\
&= \underbrace{\int \pi_\theta(x) \nabla_\theta \log \pi_\theta(x) f(x) \, dx}_{\text{Log-Derivative Trick}} + \underbrace{\mathbb{E}_{x \sim \pi_\theta} [\nabla_\theta f(x)]}_{\text{你公式的右边}} \\
&= \mathbb{E}_{x \sim \pi_\theta} [f(x) \nabla_\theta \log \pi_\theta(x)] + \mathbb{E}_{x \sim \pi_\theta} [\nabla_\theta f(x)]
\end{aligned}
$$

- 等式右边 $\mathbb{E}_{x \sim \pi_\theta} [\nabla_\theta f(x)]$ 只是完整梯度的一部分。它丢失了第一项（通常称为 Score Function 项或 REINFORCE 项）。
- 在 RL 中，$f(x)$ 通常是奖励函数 $R(x)$。奖励函数通常由环境给出，与策略参数 $\theta$ 无关。
    - $\nabla_\theta f(x) = 0$因此，不等式右边 $\mathbb{E}[\nabla_\theta f(x)] = 0$。但不等式左边（策略梯度）显然不为 0。

### 计算图 （AutoGrad）

$$
\begin{aligned}
\text{LHS} &= \nabla_\theta \left( \sum_{x} \pi_\theta(x) \cdot f(x) \right) \\
&= \sum_{x} \nabla_\theta (\pi_\theta(x) \cdot f(x)) \\
&= \sum_{x} \underbrace{[\nabla_\theta \pi_\theta(x)] \cdot f(x)}_{\text{第一项：概率变化的影响}} + \sum_{x} \underbrace{\pi_\theta(x) \cdot [\nabla_\theta f(x)]}_{\text{第二项：函数值变化的影响}}
\end{aligned}
$$
- 第一项（Score Function Term）： 这一项捕捉的是“因为 $\theta$ 变了，所以采样到 $x$ 的概率变了，进而导致总期望变了”。这是 Policy Gradient 的核心来源。
- 第二项（Pathwise Derivative Term）： 这一项捕捉的是“对于固定的 $x$，如果 $\theta$ 变了，这个样本的得分 $f(x)$ 怎么变”。

- 正常的计算流（可导）：$\theta \xrightarrow{\text{计算}} \text{Logits} \xrightarrow{\text{Softmax}} \text{Prob} \xrightarrow{\text{计算}} \text{Loss}$
- RL 的计算流（含采样）：$\theta \xrightarrow{\text{计算}} \text{Prob} \xrightarrow[\color{red}{\textbf{采样动作}}]{\text{从概率分布中摇出一个 token } x} x \xrightarrow{\text{计算}} f(x)$
    -  PyTorch 执行 `Categorical(probs).sample()` 得到 $x$（比如 token ID = 42））
    -  采样操作是一个阶跃函数（Step Function），其导数在几乎所有地方都是 0。
- AutoGrad 的机制是**“基于样本的反向传播”**。它拿到的输入是一个已经采样出来的样本 $x$（在这个 Backward 过程中，$x$ 被视为常数）。AutoGrad 只能看到 $f(x)$ 这个计算图内部的 $\theta$。
    - $\text{RHS} = \mathbb{E}_{x \sim \pi_\theta} [\nabla_\theta f(x)] = \sum_{x} \pi_\theta(x) \cdot [\nabla_\theta f(x)]$
- AutoGrad 丢失的第一项是：$\sum_{x} [\nabla_\theta \pi_\theta(x)] \cdot f(x)$
    - 在监督学习（Supervised Learning）中，数据分布是固定的（来自数据集），$\pi(x)$ 不依赖 $\theta$，所以这一项是 0，AutoGrad 没错。
    - 但在 RL 中，数据是你自己生成的（Self-generated）。如果你想最大化奖励，你主要靠的是**“让高分样本出现的概率变大”**（即改变 $\pi_\theta(x)$），而不是“改变样本的分数”。AutoGrad 的盲区：AutoGrad 看着样本 $x$，心想：“这是一个固定的数据”。它不知道这个 $x$ 是从 $\pi_\theta$ 里摇出来的。如果你不显式地告诉它（通过 REINFORCE/Log-Derivative Trick），它就无法对采样概率求导。

### REINFORCE

> 对数导数技巧” (Log-Derivative Trick)。只能从 $\pi$ 采样，不能从 $\nabla \pi$ 采样

$$
\text{Missing Term} = \sum_{\text{all possible } x} \nabla_\theta \pi_\theta(x) \cdot f(x)
$$

- AutoGrad 只能对采样出来的样本求导。也就是它只能处理 $\mathbb{E}_{x \sim \pi}[\dots]$ 这种形式。困难点： 上面的公式里，$\nabla_\theta \pi_\theta(x)$ 没有乘 $\pi_\theta(x)$，所以它不是一个期望（Expectation）形式。它只是一个普通的求和。计算机无法通过“采样 $x$”来近似这个求和。

$$
\nabla_\theta \pi_\theta(x) = \pi_\theta(x) \cdot \nabla_\theta \log \pi_\theta(x)
$$

$$
\begin{aligned}
\text{Missing Term} &= \sum_{x} \mathbf{\nabla_\theta \pi_\theta(x)} \cdot f(x) \\
&= \sum_{x} \underbrace{\mathbf{\pi_\theta(x) \cdot \nabla_\theta \log \pi_\theta(x)}}_{\text{替换这一部分}} \cdot f(x) \\
&= \sum_{x} \pi_\theta(x) \cdot [\nabla_\theta \log \pi_\theta(x) \cdot f(x)]\\
&= \mathbb{E}_{x \sim \pi_\theta} [\nabla_\theta \log \pi_\theta(x) \cdot f(x)]
\end{aligned}
$$

- $\nabla \pi$ 不行：因为你不能问“如果我参数变一点，抽出来的这个人会变成什么样？”（采样不可导）。
    - $\nabla \pi$：是一个数值，但不是分布，无法用于采样。
- $\nabla \log \pi \cdot f(x)$ 行：因为你在问“如果我参数变一点，抽中这个人的概率会怎么变？”（概率是可导的）。而 $f(x)$ 只是一个权重，告诉你是该把这个概率往高了拉，还是往低了踩。

### surrogate loss

既然我们推导出了真理的梯度方向应该是 $\nabla_\theta \log \pi_\theta(x) \cdot f(x)$，我们需要构造一个假的 Loss，让 AutoGrad 自动算出这个结果。

$$
L_{surrogate} = - \log \pi_\theta(x) \cdot \text{detach}(f(x))
$$

当我们对这个 $L_{surrogate}$ 执行 .backward() 时，AutoGrad 会做以下计算：它把 detach(f(x)) 当作一个常数系数（比如叫 $R$）。它计算 $\log \pi_\theta(x)$ 关于 $\theta$ 的梯度。它把两者乘起来。结果就是：$$\nabla_\theta L_{surrogate} = - f(x) \cdot \nabla_\theta \log \pi_\theta(x)$$